# Local EDA 

**Work outline**
- Review resampled 10 min vs. aligned 10 min resample

In [26]:
#Import relevant libraries 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

from scipy.stats import skew, kurtosis, ks_2samp

import warnings
warnings.filterwarnings('ignore')

#Display settings 
pd.set_option('display.max_columns', None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [10]:
#Upload csv files 

#10-min resampled 
df_10T = pd.read_csv("~/Desktop/Coding/cive70088/EDS_Design_Project/data/df_10min.csv") 

#10-min resampled (aligned)
df_aligned_10T = pd.read_csv("~/Desktop/Coding/cive70088/EDS_Design_Project/data/df_aligned_10T.csv") 

## Statistical Analysis of Resampled Data Frames 

Compare 10 min resampled dataframes (timestamps aligned and not aligned) to the original dataframe

In [11]:
#Check shape of dataframes 
print("Shape of 10-min resampled dataframe:", df_10T.shape)
print("Shape of 10-min resampled dataframe (shifted timestamps):", df_aligned_10T.shape)

Shape of 10-min resampled dataframe: (105264, 35)
Shape of 10-min resampled dataframe (shifted timestamps): (105264, 35)


In [27]:
#Define key features for analysis
features = ['inflow', 't1_airflow', 't1_n2o', 't1_nh4','t1_no3', 't1_do', 't1_temp', 't1_po4',
             't1_ss', 't1_valve_pct']

In [29]:

def compare_two_datasets(df1, df2, cols=None, name1="DF1", name2="DF2"):

    # Automatically detect shared numeric columns
    if cols is None:
        num1 = df1.select_dtypes(include="number").columns
        num2 = df2.select_dtypes(include="number").columns
        cols = list(set(num1).intersection(num2))

    results = []

    for col in cols:

        s1 = df1[col].dropna()
        s2 = df2[col].dropna()

        if len(s1) == 0 or len(s2) == 0:
            continue

        # Basic stats
        mean1, mean2 = s1.mean(), s2.mean()
        std1, std2 = s1.std(), s2.std()

        # Robust spread
        iqr1 = s1.quantile(0.75) - s1.quantile(0.25)
        iqr2 = s2.quantile(0.75) - s2.quantile(0.25)

        # KS test (distribution similarity)
        ks_stat, ks_p = ks_2samp(s1, s2)

        results.append({
            "variable": col,

            f"{name1}_rows": len(df1),
            f"{name2}_rows": len(df2),

            f"{name1}_mean": mean1,
            f"{name2}_mean": mean2,
            "mean_diff": mean2 - mean1,

            f"{name1}_median": s1.median(),
            f"{name2}_median": s2.median(),

            f"{name1}_std": std1,
            f"{name2}_std": std2,
            "std_ratio": std2 / std1 if std1 != 0 else np.nan,

            f"{name1}_IQR": iqr1,
            f"{name2}_IQR": iqr2,

            f"{name1}_skew": skew(s1),
            f"{name2}_skew": skew(s2),

            f"{name1}_kurtosis": kurtosis(s1),
            f"{name2}_kurtosis": kurtosis(s2),

            f"{name1}_missing_%": df1[col].isna().mean() * 100,
            f"{name2}_missing_%": df2[col].isna().mean() * 100,

            "KS_statistic": ks_stat,
            "KS_p_value": ks_p
        })

    return pd.DataFrame(results).sort_values("KS_statistic", ascending=False)

In [30]:
comparison = compare_two_datasets(
    df_10T,
    df_aligned_10T,
    features,
    name1="10T",
    name2="10T (aligned)"
)

comparison

,variable,10T_rows,10T (aligned)_rows,10T_mean,10T (aligned)_mean,mean_diff,10T_median,10T (aligned)_median,10T_std,10T (aligned)_std,std_ratio,10T_IQR,10T (aligned)_IQR,10T_skew,10T (aligned)_skew,10T_kurtosis,10T (aligned)_kurtosis,10T_missing_%,10T (aligned)_missing_%,KS_statistic,KS_p_value
0,inflow,105264,105264,3083.477209,3083.172523,-0.304686,2319.155070,2319.097217,2187.110523,2188.031254,1.000421,2043.402734,2037.644755,1.462880,1.462291,2.230519,2.229822,13.633341,13.635241,0.002905,0.836430
4,t1_no3,105264,105264,3.116311,3.116332,0.000021,2.511774,2.511954,2.771673,2.772084,1.000148,2.535694,2.538509,6.460901,6.459194,78.816717,78.787417,1.020292,1.021242,0.001298,0.999993
1,t1_airflow,105264,105264,2187.268389,2187.190458,-0.077931,2190.054934,2187.724329,2020.493134,2020.201553,0.999856,3938.204852,3936.867152,0.277609,0.277741,-1.331575,-1.330921,10.163969,10.161119,0.001273,0.999999
5,t1_do,105264,105264,0.653175,0.653207,0.000032,0.437970,0.437862,0.569061,0.569319,1.000454,0.846336,0.846336,1.627498,1.633424,11.695298,11.750148,1.407889,1.414539,0.001226,0.999999
9,t1_valve_pct,105264,105264,44.470745,44.468659,-0.002086,43.500000,43.560001,40.266240,40.268408,1.000054,85.740001,85.779999,0.099639,0.099971,-1.677241,-1.677429,4.411765,4.419365,0.000949,1.000000
8,t1_ss,105264,105264,2.552777,2.552733,-0.000044,2.548741,2.548539,0.852093,0.852148,1.000064,0.823943,0.824772,-0.223153,-0.223000,1.280149,1.278433,1.020292,1.023142,0.000877,1.000000
7,t1_po4,105264,105264,1.124365,1.124290,-0.000075,0.891840,0.891922,0.957438,0.957453,1.000016,0.927564,0.927804,3.295983,3.298196,20.948145,20.968501,1.069691,1.072541,0.000816,1.000000
2,t1_n2o,105264,105264,0.128340,0.129019,0.000679,0.037731,0.037616,0.225140,0.239616,1.064299,0.121296,0.121528,3.479031,8.233288,16.170144,265.408573,2.664729,2.686579,0.000802,1.000000
3,t1_nh4,105264,105264,2.311437,2.311440,0.000003,1.869051,1.870586,2.200410,2.200323,0.999961,2.057673,2.061961,4.147375,4.148012,27.896563,27.905162,1.022192,1.024092,0.000791,1.000000
6,t1_temp,105264,105264,15.660482,15.660668,0.000186,15.635127,15.635648,3.381279,3.381365,1.000025,6.283004,6.283185,-0.001703,-0.001573,-1.335390,-1.334604,3.699270,3.701170,0.000160,1.000000
